In [ ]:
# test space for random forrest, SVM, and LSTM based classification for EMG data 

In [ ]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

Defining Initial Variables

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/EEG_data"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

Pre-Processing Steps

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [ ]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features 
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

  
     return var, rms, wl, fmd


Testing Classification 

In [ ]:
def make_features_df(subject_epoch,subject,block):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
            "WL_Zygo_Sum", 
            "Var_Zygo_Sum", 
            "RMS_Zygo_Sum", 
            "Power_Zygo_Sum", 
            "WL_Corr_Sum", 
            "Var_Corr_Sum", 
            "RMS_Corr_Sum",
            "Power_Corr_Sum"
        ]
        )   
    
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
        epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

        # get summary features for simple classification 
        wl_zygo = np.sum(np.abs(np.diff(epoch_zygo)))  
        var_zygo = np.var(epoch_zygo)  
        rms_zygo  =  np.sqrt((1/len(epoch_zygo))*np.sum(epoch_zygo**2))  
        power_zygo = np.mean(epoch_zygo**2)
 
        wl_corr = np.sum(np.abs(np.diff(epoch_corr)))  
        var_corr = np.var(epoch_corr)  
        rms_corr  =  np.sqrt((1/len(epoch_corr))*np.sum(epoch_corr**2))  
        power_corr = np.mean(epoch_corr**2)

        # fill dataframe 
        features.loc[t] = [
            subject, 
            int(block),
            t + 1,
            subject_epoch[t].metadata['True_activation'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_wl_zygo,
            epoch_var_zygo, 
            epoch_rms_zygo, 
            epoch_wl_corr,
            epoch_var_corr, 
            epoch_rms_corr, 
            wl_zygo,
            var_zygo,
            rms_zygo, 
            power_zygo, 
            wl_corr, 
            var_corr,
            rms_corr, 
            power_corr, 
        ]
    
    return features 

In [ ]:
# extracting features for classification 
i = 0 
features_results_mat = []

for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]
            print(file)
            print(subject+block)


            # need to leave out these subjects 
            if (subject == "NL02IF" or 
                subject == "NL05WW" or 
                subject == "NL01SS" or 
                subject == "RL11JH" or 
                subject == "RL12JL" or 
                subject == "RL07BR"):
                continue

            subject_epoch, _ = pre_process_subjets(subject,block)
            i += 1
            # make features data frame for each subject
            features = make_features_df(subject_epoch,subject,block)
            features_results_mat.append(features)


print(f"{i} Subjects & Naps Processed")

In [ ]:
print(i)

In [ ]:
features_all = pd.concat(features_results_mat, ignore_index=True)

In [ ]:
print(np.shape(features_results_mat))

Testing Classification 

Random Forrest - Test with Classification from all 6 Features and Single Muscle Activation 

In [ ]:

X = features_all[["WL_Zygo_Sum", 
            "Var_Zygo_Sum", 
            "RMS_Zygo_Sum", 
            "Power_Zygo_Sum", 
            "WL_Corr_Sum", 
            "Var_Corr_Sum", 
            "RMS_Corr_Sum",
            "Power_Corr_Sum"]]  
y = features_all["True_Muscle_Activated"]       

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # good for classification if classes are imbalanced
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # helps if classes are imbalanced; remove if not needed
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Classes
classes = ['Corr', 'None', 'Zygo']

# Metrics
precision = [0.85, 0.88, 0.81]
recall = [0.77, 0.88, 0.86]
f1 = [0.81, 0.88, 0.84]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots()

bars1 = ax.bar(x - width, precision, width, label='Precision')
bars2 = ax.bar(x, recall, width, label='Recall')
bars3 = ax.bar(x + width, f1, width, label='F1-score')

# Labels and formatting
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Classification on Muscle Activation')
ax.legend()

# Add values on bars
ax.bar_label(bars1, fmt='%.2f')
ax.bar_label(bars2, fmt='%.2f')
ax.bar_label(bars3, fmt='%.2f')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

Random Forrest - Test with Classification for number of contractions 

In [ ]:
# random forrest for Zygo  
X_zygo = features_all[["WL_Zygo_Sum", 
            "Var_Zygo_Sum", 
            "RMS_Zygo_Sum", 
            "Power_Zygo_Sum"]]  
y_zygo = features_all["Num_Contractions_Zygo"]      
y_zygo = np.where(y_zygo > 3, 1, 2) # class 2 less than 3 

X_train_zygo, X_test_zygo, y_train_zygo, y_test_zygo = train_test_split(
    X_zygo, y_zygo,
    test_size=0.2,
    random_state=42,
    stratify=y_zygo  # good for classification if classes are imbalanced
)

rf_zygo = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # helps if classes are imbalanced; remove if not needed
)

rf_zygo.fit(X_train_zygo, y_train_zygo)

y_pred_zygo = rf_zygo.predict(X_test_zygo)

print("Accuracy Zygo:", accuracy_score(y_test_zygo, y_pred_zygo))
print("\nConfusion matrix Zygo:\n", confusion_matrix(y_test_zygo, y_pred_zygo))
print("\nReport Zygo:\n", classification_report(y_test_zygo, y_pred_zygo))

# random forrest for Corr 
X_corr= features_all[["WL_Corr_Sum", 
            "Var_Corr_Sum", 
            "RMS_Corr_Sum",
            "Power_Corr_Sum"]]
y_corr = features_all["Num_Contractions_Corr"]      
y_corr = np.where(y_corr > 3, 1, 2)

X_train_corr, X_test_corr, y_train_corr, y_test_corr = train_test_split(
    X_corr, y_corr,
    test_size=0.2,
    random_state=42,
    stratify=y_corr  # good for classification if classes are imbalanced
)

rf_corr = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # helps if classes are imbalanced; remove if not needed
)

rf_corr.fit(X_train_corr, y_train_corr)

y_pred_corr = rf_corr.predict(X_test_corr)

print("Accuracy Corr:", accuracy_score(y_test_corr, y_pred_corr))
print("\nConfusion matrix Corr:\n", confusion_matrix(y_test_corr, y_pred_corr))
print("\nReport Corr:\n", classification_report(y_test_corr, y_pred_corr))

In [ ]:
for t in range(60):
    subject_info = features_all.loc[t,["Subject"]].iloc[0]
    nap_info = features_all.loc[t,["Nap Number"]].iloc[0]

   # print(subject_info,nap_info)

    sample = features_all.loc[t,["WL_Zygo_Sum", 
            "Var_Zygo_Sum", 
            "RMS_Zygo_Sum", 
            "Power_Zygo_Sum"]]
    sample = sample.to_numpy()
    sample = sample.reshape(1,4)
    #sample = np.array(sample.tolist()).reshape(1, 2251, 2)

    prediction = rf_zygo.predict(sample)
    print("Num Zygo:",int(features_all.loc[t, ["Num_Contractions_Zygo"]].iloc[0]),"Prediction Value:",prediction)
    #print("Predicted Zygo Array: ",prediction)
    print("Epoch: ",t+1)
    print("")




In [ ]:

# Classes
classes = ['<3', '>=3']

# Metrics
precision = [0.25, 0.99]
recall = [0.08, 1.00]
f1 = [0.12, 0.99]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots()

bars1 = ax.bar(x - width, precision, width, label='Precision')
bars2 = ax.bar(x, recall, width, label='Recall')
bars3 = ax.bar(x + width, f1, width, label='F1-score')

# Labels and formatting
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_xlabel('Number of Contractions')
ax.set_title('Classification on Num Zygo Contractions')
ax.legend()

# Add values on bars
ax.bar_label(bars1, fmt='%.2f')
ax.bar_label(bars2, fmt='%.2f')
ax.bar_label(bars3, fmt='%.2f')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

Testing CNN (takes continuous signal)
https://github.com/Abhishek-Atole/Classification-of-an-EMG-Signal-Using-CNN/blob/main/Classification_of_EMG_Signal.ipynb

In [ ]:
from tensorflow.keras.models import Sequential  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input  # Import necessary layers from TensorFlow Keras

def CNN_model(input_shape, num_classes,feature_num):
    model = Sequential([
        Input(shape=(2251, feature_num))
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers

    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu'))  # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation

    # Compile the model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # Compile the model with Adam optimizer, sparse categorical crossentropy loss, and accuracy metric

    # Print model summary
    model.summary()  # Print model summary

    return model  # Return the compiled model

In [ ]:
# Define CNN model
X = features_all[[ "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "Var_Corr",
            "RMS_Corr", 
            "WL_Corr"]]  

X = X.to_numpy()
X = np.array(X.tolist())
X = np.transpose(X, (0, 2, 1))

Y = features_all[["True_Muscle_Activated"]] 
Y = (Y == "Zygo").astype(int).to_numpy()    
 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

input_shape = X_train.shape[1:]  # Determine input shape based on the processed data with RMS
num_classes = len(np.unique(Y))   # Determine the number of classes based on unique values in the target vector

# Adjust input shape to match the expected format for Conv1D layer
input_shape = (input_shape[0], 1)  # Convert input shape to (input_shape[0], 1)

# Create CNN model using the adjusted input shape and number of classes
model = CNN_model(input_shape, num_classes)

history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score  # Import necessary metrics

# Predict on training data
y_pred_train = model.predict(X_train)  # Make predictions on training data
y_pred_train = np.argmax(y_pred_train, axis=1)  # Convert predictions from one-hot encoding to class labels

# Predict on test data
y_pred_test = model.predict(X_test)  # Make predictions on test data
y_pred_test = np.argmax(y_pred_test, axis=1)  # Convert predictions from one-hot encoding to class labels

# Calculate accuracy
accuracy_training = accuracy_score(y_train, y_pred_train)  # Calculate accuracy on training data
accuracy_test = accuracy_score(y_test, y_pred_test)  # Calculate accuracy on test data

# Calculate F1 score
f1_training = f1_score(y_train, y_pred_train, average='weighted')  # Calculate F1 score on training data
f1_test = f1_score(y_test, y_pred_test, average='weighted')  # Calculate F1 score on test data

# Print accuracy and F1 score
print("Training Accuracy RMS:", accuracy_training)  # Print training accuracy
print("Test Accuracy RMS:", accuracy_test)  # Print test accuracy
print("Training F1 Score RMS:", f1_training)  # Print training F1 score
print("Test F1 Score RMS:", f1_test)  # Print test F1 score

In [ ]:
from sklearn.metrics import classification_report  # Import classification_report function
# Generate classification report including accuracy and F1 score for each class
report = classification_report(y_test, y_pred_test, target_names=[str(label) for label in np.unique(y_test)])

# Print the classification report
print(report)

Testing CNN on Classifying Number of Activations for Zygo 

In [ ]:
# Define CNN model
X = features_all[[ "WL_Zygo", # three features being used 
            "Var_Zygo"]]

X = X.to_numpy()
X = np.array(X.tolist())
X = np.transpose(X, (0, 2, 1))

Y = features_all["Num_Contractions_Zygo"].astype(int).to_numpy()      
   
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

input_shape = X_train.shape[1:]  # Determine input shape based on the processed data with RMS
num_classes = len(np.unique(Y))   # Determine the number of classes based on unique values in the target vector

# Adjust input shape to match the expected format for Conv1D layer
input_shape = (input_shape[0], 1)  # Convert input shape to (input_shape[0], 1)

# Create CNN model using the adjusted input shape and number of classes
model = CNN_model(input_shape, num_classes,2)

history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score  # Import necessary metrics

model_zygo = model 
# Predict on training data
y_pred_train = model_zygo.predict(X_train)  # Make predictions on training data
y_pred_train = np.argmax(y_pred_train, axis=1)  # Convert predictions from one-hot encoding to class labels

# Predict on test data
y_pred_test = model_zygo.predict(X_test)  # Make predictions on test data
y_pred_test = np.argmax(y_pred_test, axis=1)  # Convert predictions from one-hot encoding to class labels

# Calculate accuracy
accuracy_training = accuracy_score(y_train, y_pred_train)  # Calculate accuracy on training data
accuracy_test = accuracy_score(y_test, y_pred_test)  # Calculate accuracy on test data

# Calculate F1 score
f1_training = f1_score(y_train, y_pred_train, average='weighted')  # Calculate F1 score on training data
f1_test = f1_score(y_test, y_pred_test, average='weighted')  # Calculate F1 score on test data

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training)  # Print training accuracy
print("Test Accuracy :", accuracy_test)  # Print test accuracy
print("Training F1 Score :", f1_training)  # Print training F1 score
print("Test F1 Score :", f1_test)  # Print test F1 score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
metrics = ['Accuracy', 'F1 Score']
training = [0.9105555555555556, 0.9006149893779598]
test = [0.8566666666666667, 0.834443375315734]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots()

bars1 = ax.bar(x - width/2, training, width, label='Training')
bars2 = ax.bar(x + width/2, test, width, label='Test')

# Axis labels
ax.set_ylabel('Score')
ax.set_title('Training vs Test Performance (Corr)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.legend()

# Write values on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
            f'{height:.3f}', ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
            f'{height:.3f}', ha='center', va='bottom')

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

In [ ]:
for t in range(60):
    subject_info = features_all.loc[t,["Subject"]].iloc[0]
    nap_info = features_all.loc[t,["Nap Number"]].iloc[0]

   # print(subject_info,nap_info)

    sample = features_all.loc[t,["WL_Zygo","Var_Zygo"]]
    sample = sample.to_numpy()
    sample = np.array(sample.tolist()).reshape(1, 2251, 2)

    prediction = model_zygo.predict(sample, verbose=0)

    if (int(features_all.loc[t, ["Num_Contractions_Zygo"]].iloc[0]) != np.argmax(prediction, axis=1)[0]):
        print("Num Zygo:",int(features_all.loc[t, ["Num_Contractions_Zygo"]].iloc[0]),"Prediction Value:",np.argmax(prediction, axis=1)[0])
#print("Predicted Zygo Array: ",prediction)
        print("Epoch: ",t+1)
        print("")




In [ ]:
report = classification_report(y_test, y_pred_test, target_names=[str(label) for label in np.unique(y_test)])

# Print the classification report
print(report)

Testing CNN on Classifying Number of Activations for Corr 

In [ ]:
# Define CNN model
X = features_all[[ "WL_Corr",
                  "Var_Corr"]]

X = X.to_numpy()
X = np.array(X.tolist())
X = np.transpose(X, (0, 2, 1))

Y = features_all["Num_Contractions_Corr"].astype(int).to_numpy()    
Y[Y > 9] = 8 # force a number of contractions greater than 8 to 9

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

input_shape = X_train.shape[1:]  # Determine input shape based on the processed data with RMS
num_classes = len(np.unique(Y))   # Determine the number of classes based on unique values in the target vector

# Adjust input shape to match the expected format for Conv1D layer
input_shape = (input_shape[0], 1)  # Convert input shape to (input_shape[0], 1)

# Create CNN model using the adjusted input shape and number of classes
model = CNN_model(input_shape, num_classes,2)
model_corr = model 
history = model.fit(X_train, y_train, epochs=9, validation_data=(X_test, y_test))

In [ ]:
# Predict on training data
from sklearn.metrics import accuracy_score, f1_score  # Import necessary metrics

y_pred_train = model.predict(X_train)  # Make predictions on training data
y_pred_train = np.argmax(y_pred_train, axis=1)  # Convert predictions from one-hot encoding to class labels

# Predict on test data
y_pred_test = model.predict(X_test)  # Make predictions on test data
y_pred_test = np.argmax(y_pred_test, axis=1)  # Convert predictions from one-hot encoding to class labels

# Calculate accuracy
accuracy_training = accuracy_score(y_train, y_pred_train)  # Calculate accuracy on training data
accuracy_test = accuracy_score(y_test, y_pred_test)  # Calculate accuracy on test data

# Calculate F1 score
f1_training = f1_score(y_train, y_pred_train, average='weighted')  # Calculate F1 score on training data
f1_test = f1_score(y_test, y_pred_test, average='weighted')  # Calculate F1 score on test data

# Print accuracy and F1 score
print("Training Accuracy RMS:", accuracy_training)  # Print training accuracy
print("Test Accuracy RMS:", accuracy_test)  # Print test accuracy
print("Training F1 Score RMS:", f1_training)  # Print training F1 score
print("Test F1 Score RMS:", f1_test)  # Print test F1 score

In [ ]:
for t in range(60):
    subject_info = features_all.loc[t,["Subject"]].iloc[0]
    nap_info = features_all.loc[t,["Nap Number"]].iloc[0]

   # print(subject_info,nap_info)

    sample = features_all.loc[t,["WL_Corr","Var_Corr"]]
    sample = sample.to_numpy()
    sample = np.array(sample.tolist()).reshape(1, 2251,2)

    prediction = model.predict(sample, verbose=0)

    if (int(features_all.loc[t, ["Num_Contractions_Corr"]].iloc[0]) != np.argmax(prediction, axis=1)[0]):
        print("Num Corr:",int(features_all.loc[t, ["Num_Contractions_Corr"]].iloc[0]),"Prediction Value:",np.argmax(prediction, axis=1)[0])
#print("Predicted Zygo Array: ",prediction)
        print("Epoch: ",t+1)
        print("")

